# Balance de materia y energía de una planta de tratamiento de crudo

El balance de materia y energía se desarrollará siguiente el PFD que se presenta acontinuación
![PFD.png](PFD.png)

## Consideraciones

1. No hay acumulaciones o perdidas en los equipos
2. El proceso se encuentra en estado estacionario
3. La composición de las corrientes solo contiene Crudo (c) y Agua (a)


## 1.Balance de matería del proceso
Para poder resolver el balance de matería del proceso se propone el siguiente PFD simplificado:
![PFD_s.png](PFD_s.png)

### 1.1. Conversión del flujo en la corriente 1
La descricpción dle proceso indica que la corriente uno tiene un flujo de 117,4 kBPD (miles de barriles por día), por consiguiente se procede a hacer la conversión a ton/día. Para esto primero se determina la densidad de la corriente ($\rho_1$):

$\rho_1 = \rho_c \frac{(X_{c,1}/\rho_c)}{(X_{c,1}/\rho_c)+(X_{a,1}/\rho_a)} + \rho_a \frac{(X_{a,1}/\rho_a)}{(X_{c,1}/\rho_c)+(X_{a,1}/\rho_a)}$

Lo primero será determinar la densidad del crudo y agua a las condiciones de la corrinte 1:

$\rho_a = 999.65 + 0.20438 T - 0.06174T^{1.2}$

$\rho_c = 1000(-0.000331*((1.8T+32)-60)+\frac{\rho_{cs}}{1000})$


In [10]:
def f_ri(T):
    rcs = 910 #Kg/m3
    ra = round(999.65+(0.20438*T)-(0.06174*(T**1.5)),2) #Kg/m3
    rc = round(1000*((rcs/1000)-(0.000331*(((1.8*T)+32)-60))),2) #Kg/m3
    return ra,rc

def f_r (Xa, Xc,ra,rc):
    r = round((rc*((Xc/rc)/((Xc/rc)+(Xa/ra))))+(ra*((Xa/ra)/((Xc/rc)+(Xa/ra)))),2) #Kg/m3
    return r
    
def X (BSW):
    ra = 1000 #Kg/m3
    rc = 910 #Kg/m3
    Xa = round((BSW*ra)/((BSW*ra)+((1-BSW)*rc)),4)
    Xc = round(1-Xa,4)
    return Xa,Xc

def f_Cp (T,Xa,Xc):
    rcs = 910 #Kg/m3
    Cc = 4.18651*((rcs/1000)**-0.5)*(0.388+(0.00045*((1.8*T)+32))) #KJ/Kg.K
    Ca = 4.1816+(0.00036*(T-20)) #KJ/Kg.K
    Cp = round(((Ca*Xa)+(Cc*Xc))*1000,2) #KJ/ton.K
    return Cp
    

In [11]:
T1 = round((145-32)*(5/9),2) #°C
ra1 = f_ri(T1)[0]
rc1 = f_ri(T1)[1]
BSW1 = 0.937
Xa1 = X(BSW1)[0]
Xc1 = X(BSW1)[1]
r1 = f_r(Xa1,Xc1,ra1,rc1)
print("Densidad 1 = ",r1," Kg/m3 @",T1)

Densidad 1 =  975.39  Kg/m3 @ 62.78


#### Conociendo la densidad de la corriente 1, se procede con la conversión del flujo:

$M_1 = 117400 \frac{bbl}{d} (\frac{0.1589m^3}{1 bbl})(\frac{993.81 Kg}{1m^3})(\frac{1 ton}{1000 Kg})$

In [12]:
V1 = 117.4*1000 #bbl/d
M1 = round((V1*r1*0.158)/1000,2) #ton/d
print("M1 = ",M1," ton/d  ///  Xa1 = ",Xa1," /// Xc1 = ",Xc1)

M1 =  18092.7  ton/d  ///  Xa1 =  0.9423  /// Xc1 =  0.0577


### 1.2. Balance de matería Mezclador 1 (M1)

In [13]:
M19m = 1480 #ton/d
T19 = 77.14-15 #°C
ra19 = f_ri(T19)[0]
rc19 = f_ri(T19)[1]
BSW19 = 0.4
Xa19 = X(BSW19)[0]
Xc19 = X(BSW19)[1]
r19 = f_r(Xa19,Xc19,ra19,rc19)
#----------------------------------
M2 = round(M1 + M19m,2) #ton/d
Xa2 = round(((M1*Xa1)+(M19m*Xa19))/M2,4)
Xc2 = round(1-Xa2,4)
print("M2 = ",M2," ton/d  ///  Xa2 = ",Xa2," /// Xc2 = ",Xc2)
print("M19 = ",M19m," ton/d  ///  Xa19 = ",Xa19," /// Xc19 = ",Xc19)

M2 =  19572.7  ton/d  ///  Xa2 =  0.903  /// Xc2 =  0.097
M19 =  1480  ton/d  ///  Xa19 =  0.4228  /// Xc19 =  0.5772


In [14]:
#Se determina la temperatura de la corriente 2
Cp1 = f_Cp(T1,Xa1,Xc1)  #KJ/ton.K
Cp19 = f_Cp(T19,Xa19,Xc19)  #KJ/ton.K
T1_k = T1 + 273.15 #K
T19_k = T19 + 273.15 #K
T2 = round((((M1*T1_k*Cp1)+(M19m*T19_k*Cp19))/((M1*Cp1)+(M19m*Cp19)))-273.15,2)
print("Temperatura de la corriente 2 = ",T2,"°C")

Temperatura de la corriente 2 =  62.74 °C


### 1.3. Balance del separador TK-201

In [15]:
#Definimos el flujo 3
M3 = M2 #ton/d
T3 = round((174-32)*(5/9),2) #°C
Xa3 = Xa2
Xc3 = Xc2
ra3 = f_ri(T3)[0]
rc3 = f_ri(T3)[1]
r3 = f_r(Xa3,Xc3,ra3,rc3)
print("M3 = ",M3," ton/d  ///  Xa3 = ",Xa3," /// Xc3 = ",Xc3,"///  r3 = ",r3," Kg/m3 @ ",T3)

M3 =  19572.7  ton/d  ///  Xa3 =  0.903  /// Xc3 =  0.097 ///  r3 =  961.79  Kg/m3 @  78.89


In [16]:
V4 = 20.92*1000 #bbl/d
T4 = round((170-32)*(5/9),2) #°C
ra4 = f_ri(T4)[0]
rc4 = f_ri(T4)[1]
BSW4 = 0.35
Xa4 = X(BSW4)[0]
Xc4 = X(BSW4)[1]
r4 = f_r(Xa4,Xc4,ra4,rc4)
M4 = round((V4*r4*0.1589)/1000,2) #ton/d
#---------------------------------------
M11 = M3 - M4 #ton/d
Xa11 = round(((M3*Xa3)-(M4*Xa4))/M11,4)
Xc11 = round(((M3*Xc3)-(M4*Xc4))/M11,4)
print("M4 = ",M4," ton/d  ///  Xa4 = ",Xa4," /// Xc4 = ",Xc4,"///  r4 = ",r4," Kg/m3 @ ",T4)
print("M11 = ",M11," ton/d  ///  Xa11 = ",Xa11," /// Xc11 = ",Xc11)

M4 =  3019.56  ton/d  ///  Xa4 =  0.3717  /// Xc4 =  0.6283 ///  r4 =  908.36  Kg/m3 @  76.67
M11 =  16553.14  ton/d  ///  Xa11 =  0.9999  /// Xc11 =  0.0001


### 1.4. Balance del separador TK-202

In [17]:
V7 = 7.40*1000 #bbl/d
T7 = round((180-32)*(5/9),2) #°C
ra7 = f_ri(T7)[0]
rc7 = f_ri(T7)[1]
BSW7 = 0.005
Xa7 = X(BSW7)[0]
Xc7 = X(BSW7)[1]
r7 = f_r(Xa7,Xc7,ra7,rc7)
M7 = round((V7*r7*0.158)/1000,2) #ton/d
#------------------------------------------
M13 = M4 - M7 #ton/d
Xa13 = round(((M4*Xa4)-(M7*Xa7))/M13,2)
Xc13 = round(1-Xa13,2)
print("M7 = ",M7," ton/d  ///  Xa7 = ",Xa7," /// Xc7 = ",Xc7,"///  r7 = ",r7," Kg/m3 @ ",T7)
print("M13 = ",M13," ton/d  ///  Xa13 = ",Xa13," /// Xc13 = ",Xc13)

M7 =  1018.1  ton/d  ///  Xa7 =  0.0055  /// Xc7 =  0.9945 ///  r7 =  870.77  Kg/m3 @  82.22
M13 =  2001.46  ton/d  ///  Xa13 =  0.56  /// Xc13 =  0.44


### 1.5. Balance del mezclador 2 (M2)

In [18]:
M15 = M11 + M13 #ton/d
Xa15 = round(((M11*Xa11)+(M13*Xa13))/M15,2)
Xc15 = round(1-Xa15,2)
print("M15 = ",M15," ton/d  ///  Xa15 = ",Xa15," /// Xc15 = ",Xc15)

M15 =  18554.6  ton/d  ///  Xa15 =  0.95  /// Xc15 =  0.05


### 1.6. Balance global

In [19]:
M16 = round(M1 - M7,2) #ton/d
T16 = T19 #°C
ra16 = f_ri(T16)[0]
rc16 = f_ri(T16)[1]
BSW16 = 5/1000000
Xc16 = X(BSW16)[0]
Xa16 = X(BSW16)[1]
r16 = f_r(Xa16,Xc16,ra16,rc16)
print("M16 = ",M16," ton/d  ///  Xa16 = ",Xa16," /// Xc16 = ",Xc16,"///  r16 = ",r16," Kg/m3 @ ",T16)

M16 =  17074.6  ton/d  ///  Xa16 =  1.0  /// Xc16 =  0.0 ///  r16 =  982.11  Kg/m3 @  62.14


### 1.7. Balance de la PTAP

In [20]:
M19 = round(M15 - M16,2) #ton/d
Xa19m = round(((M15*Xa15)-(M16*Xa16))/M19,2)
Xc19m = round(1-Xa19m,2)
print("M19 = ",M19," ton/d  ///  Xa19 = ",Xa19," /// Xc19 = ",Xc19,"///  r19 = ",r19," Kg/m3 @ ",T19)

M19 =  1480.0  ton/d  ///  Xa19 =  0.4228  /// Xc19 =  0.5772 ///  r19 =  921.88  Kg/m3 @  62.14


**Debido a que los flujos y las composiciones de M19m y M19 son muy cercanos, se concluye el balance de materia**

## 2. Balance de energía

### 2.2. Balance del calentador H-202

In [21]:
Cp2 = f_Cp(T2,Xa2,Xc2) #KJ/ton.K
T3 = round((174-32)*(5/9),2) #°C
Q_202 = round((Cp2*M2*(T3-T2))/1000,2) #MJ
print("Calor suministrado en el H-202 = ",Q_202, "MJ/d")

Calor suministrado en el H-202 =  1258962.68 MJ/d


### 2.2. Balance del calentador H-203

In [22]:
T5 = round((170-32)*(5/9),2) #°C
M5 = M4 #ton/d
Xa5 = Xa4
Xc5 = Xc4
T6 = round((187-32)*(5/9),2) #°C
Cp5 = f_Cp(T5,Xa5,Xc5) #KJ/ton.K
Q_203 = round((Cp5*M5*(T6-T5))/1000,2) #MJ
print("Calor suministrado en el H-203 = ",Q_203, "MJ/d")

Calor suministrado en el H-203 =  81030.16 MJ/d


### 2.3. Calculo de la potencia consumida por las bombas

Para determinar la potencia que consumen las bombas para mover las correintes a las presiones indicadas, se hace uso de la ecuación de Ecuación de Bernoulli modificada:

$P_h = M[\frac{P_s-P_e}{\rho}+g(z_1 - Z_2)+\frac{v_{2}^{2}-v_{1}^{2}}{2}]$

Asumiendo que no hay cambio de alturas o velocidades significativas, la potencia por la bomba se determina así:

$P = \frac{M(P_s - P_e)}{\rho\eta}$

#### 2.3.1. Potencia consumida por la P-201A-C

In [23]:
n_cp = 0.8
P4 = 3 #psig
M4_k = M4*0.01157407407 #Kg/s
P5 = 45 #psig
DP_201 = (P5-P4)*6894.76 #Pa
P_201 = round(((M4_k*DP_201)/(r4*n_cp))/1000,2) #KW
print("Potencia consumida por la P-201 = ",P_201,"KW")

Potencia consumida por la P-201 =  13.93 KW


#### 2.3.2. Potencia consumida por la P-202A-F

In [24]:
n_c = 0.7
P11 = 3 #psig
T11 = T4 #°C
M11_k = M11*0.01157407407 #Kg/s
ra11 = f_ri(T11)[0] #Kg/m3
rc11 = f_ri(T11)[1] #Kg/m3
r11 = f_r(Xa11,Xc11,ra11,rc11) #Kg/m3
P12 = 20 #psig
DP_202 = (P12-P11)*6894.76 #Pa
P_202 = round(((M11_k*DP_202)/(r11*n_c))/1000,2) #kW
print("Potencia consumida por la P-202 = ",P_202,"KW")

Potencia consumida por la P-202 =  32.94 KW


#### 2.3.3. Potencia consumida por la P-203A-C

In [25]:
P13 = 3 #psig
T13 = T7 #°C
M13_k = M13*0.01157407407 #Kg/s
ra13 = f_ri(T13)[0] #Kg/m3
rc13 = f_ri(T13)[1] #Kg/m3
r13 = f_r(Xa13,Xc13,ra13,rc13) #Kg/m3
P14 = 20 #psig
DP_203 = (P14-P13)*6894.76 #Pa
P_203 = round(((M13_k*DP_203)/(r13*n_c))/1000,2) #kW
print("Potencia consumida por la P-203 = ",P_203,"KW")

Potencia consumida por la P-203 =  4.2 KW


#### 2.3.4. Potencia consumida por la P-204A-D

In [26]:
P8 = 3 #psig
T8 = T7-16.7 #°C
M8_k = M7*0.01157407407 #Kg/s
Xa8 = Xa7
Xc8 = Xc7
ra8 = f_ri(T8)[0] #Kg/m3
rc8 = f_ri(T8)[1] #Kg/m3
r8 = f_r(Xa8,Xc8,ra8,rc8) #Kg/m3
P9 = 100 #psig
DP_204 = (P9-P8)*6894.76 #Pa
P_204 = round(((M8_k*DP_204)/(r8*n_c))/1000,2) #kW
print("Potencia consumida por la P-204 = ",P_204,"KW")

Potencia consumida por la P-204 =  12.78 KW


#### 2.3.5. Potencia consumida por la P-205A-D

In [27]:
M9_k = M8_k #Kg/s
r9 = r8 #Kg/m3
P10 = 600 #psig
DP_205 = (P10-P9)*6894.76 #Pa
P_205 = round(((M9_k*DP_205)/(r9*n_c))/1000,2) #kW
print("Potencia consumida por la P-205 = ",P_205,"KW")

Potencia consumida por la P-205 =  65.89 KW


#### 2.3.6. Potencia consumida por la P-206A-D

In [28]:
M16_k = M16*0.01157407407 #Kg/s
P16 = 3 #psig
P17 = 150 #psig
DP_206 = (P17-P16)*6894.76 #Pa
P_206 = round(((M16_k*DP_206)/(r16*n_c))/1000,2) #kW
print("Potencia consumida por la P-206 = ",P_206,"KW")

Potencia consumida por la P-206 =  291.35 KW


#### 2.3.7. Potencia consumida por la P-207A-D

In [29]:
M17_k = M16_k #Kg/s
r17 = r16
P18 = 1200 #psig
DP_207 = (P18-P17)*6894.76 #Pa
P_207 = round(((M17_k*DP_207)/(r17*n_c))/1000,2) #kW
print("Potencia consumida por la P-207 = ",P_207,"KW")

Potencia consumida por la P-207 =  2081.07 KW


## 3. Indicadores

### 3.1. Consumo de energía térmica

In [30]:
CET = round(Q_202 + Q_203,2) #KJ/d
print("El consumo de energía térmica del proceso es de: ",CET," MJ/d")

El consumo de energía térmica del proceso es de:  1339992.84  MJ/d


### 3.2. Consumo de energía eléctrica

In [31]:
CEE = P_201+P_202+P_203+P_204+P_205+P_206+P_207 #kw
print("El consumo de energía eléctrica del proceso es de: ",CEE," kW")

El consumo de energía eléctrica del proceso es de:  2502.1600000000003  kW


### 3.3. Demanda de vapor

Debido a que el calor es entregado por vapor a 120 psig, por medio de tablas de vapor se determina la entalpía de evaporación ($h_{fg}$), la cual es de 2,015 kJ/kg. Conociendo este valor se procede a determinar la cantidad de vapor requerida en el proceso:

In [32]:
h_fg = 2042.73/1000 #MJ/Kg
n_rdv = 0.97
DV = round((CET/24)/(h_fg*n_rdv),2) #Kg/d
print("La demanda de vapor del proceso es de: ",DV," Kg/h")

La demanda de vapor del proceso es de:  28177.89  Kg/h


### 3.4. Demanda energética asociada a la caldera

In [33]:
MC = (12.01*32) #lbmol
MH = (1.008*52) #lbmol
PM = MC+MH
RC = MC/PM
RH = MH/PM
DQ_com = (1.8*PM)*((8400*RC)+(27765*RH)) #BTU/lbmol
DQ_com_K = (DQ_com*0.00105506)/(PM*0.453592) #Conversión a MJ/Kg
#Se determina la eficiencia de la caldera, restando las perdidas por los gases, al ambiente y humedad del combustible
K = 0.50 #Constante de Siegert para crudo pesado (21.7°API)
Tg = 365 #°F
Ta = 68 #Temperatura ambiente °F
PCO2 = 13.71 #Porcentaje de CO2 en los gases de salida (en base seca), de acuerdo con la estequeometría de la reacción de combustión
Lg = ((K*(Tg-Ta))/PCO2)/100 #Perdidas por los gases
Lp = 0.02 #Perdidas por radiación y convección hacía el ambiente
hv = 1089 #calor latente de vaporización del agua [BTU/Lb]
Cp_v = 0.46 #Calor especifico del vapor de gua [BTU/LB.°F]
Lw = (Xa7*(hv+(Cp_v*(Tg-Ta))))/DQ_com #Perdidas por la humedad del crudo combustible, corriente (7)
n_cal = round(1-Lg-Lp-Lw,3) #Eficiencia de la caldera
DEC = round((CET/24)/(n_cal*n_rdv),2) #MJ/h
print("Poder de combustión inferior del crudo: ",DQ_com_K," MJ/Kg")
print("Eficiencia de la caldera: ",n_cal*100," %")
print("Demanda energética asociada a la caldera: ",DEC," MJ/h")

Poder de combustión inferior del crudo:  44.90004471006108  MJ/Kg
Eficiencia de la caldera:  87.2  %
Demanda energética asociada a la caldera:  66008.98  MJ/h


### 3.5. Demanda de crudo combustible en la caldera

In [34]:
DC = round(((DEC*24)/DQ_com_K)/1000,2)  #ton/d
DC_b = round((DC/(r7*0.158))*1000,2) #bbl/d
print("Demanda de crudo combustible en la caldera: ",DC_b," bbl/d")

Demanda de crudo combustible en la caldera:  256.43  bbl/d


### 3.6. Demanda de combustible en el sistema de autogeneración

In [35]:
n_sa = 0.36 #Eficiencia sistema de autogeneración
CEE_k = CEE*86.4 #Conversión a MJ/d
DCSA = round((CEE_k/(DQ_com_K*n_sa))/1000,2) #ton/d
DCSA_b = round((DCSA/(r7*0.158))*1000,2) #bbl/d
print("Demanda de crudo combustible en el sistema de autogeneración: ",DCSA_b," bbl/d")

Demanda de crudo combustible en el sistema de autogeneración:  97.18  bbl/d


## 4. Indicadores de desempeño

### 4.1. Demanda de combustible por barril entregado a transporte

In [36]:
M9 = M7
T9 = T8
Xa9 = Xa8
Xc9 = Xc8
ra9 = f_ri(T9)[0] #Kg/m3
rc9 = f_ri(T9)[1] #Kg/m3
r9 = f_r(Xa9,Xc9,ra9,rc9) #Kg/m3
V9 = (M9*1000)/(r9*0.158)#bbl/d
#---------------------------------------------
Vc = (1000*(DC+DCSA))/(r9*0.158) #Volumen de crudo consumido bbl/d
Ve = V9-Vc #Volumen de crudo entregado
ICC = round((Vc/Ve)*100,2) #%
print("Indicador de consumo de crudo combustible: ",ICC," %")

Indicador de consumo de crudo combustible:  5.02  %


### 4.2. Costo energético por cada barril entregado a transporte

In [37]:
PB = 98.5 #USD/bbl
CEB = round((ICC/100)*PB,2) #USD/bbl
print("Costo energético por barril entregado: ",CEB," USD/bbl")

Costo energético por barril entregado:  4.94  USD/bbl


# 5. Consolidación de resultados

In [38]:
#Calculo de la temperatura de la corriente (15)
Cp13 = f_Cp(T13,Xa13,Xc13)  #KJ/ton.K
Cp11 = f_Cp(T11,Xa11,Xc11)  #KJ/ton.K
T11_k = T11 + 273.15 #K
T13_k = T13 + 273.15 #K
T15 = round((((M11*T11_k*Cp11)+(M13*T13_k*Cp13))/((M11*Cp11)+(M13*Cp13)))-273.15,2)
print("Temperatura de la corriente 15 = ",T15,"°C")

Temperatura de la corriente 15 =  77.15 °C


In [39]:
import pandas as pd
corrientes = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
M = [M1,M2,M2,M4,M5,M5,M7,M7,M7,M7,M11,M11,M13,M13,M15,M16,M16,M16,M19] #ton/d
T =  [T1, T2, T3, T4, T5, T6, T7, T8, T9, T9, T11, T11, T13, T13, T15, T16, T16, T16, T19] #°C
Xa = [Xa1,Xa2,Xa2,Xa4,Xa5,Xa5,Xa7,Xa7,Xa7,Xa7,Xa11,Xa11,Xa13,Xa13,Xa15,Xa16,Xa16,Xa16,Xa19]
Xc = [Xc1,Xc2,Xc2,Xc4,Xc5,Xc5,Xc7,Xc7,Xc7,Xc7,Xc11,Xc11,Xc13,Xc13,Xc15,Xc16,Xc16,Xc16,Xc19]

def f_Vol (M, r):
    V = round(M/(r*0.158),2) #KBPD
    return V

In [40]:
tabla = []

for i, c in enumerate(corrientes):
    ra, rc = f_ri(T[i])
    rm = f_r(Xa[i], Xc[i], ra, rc)
    Cp = f_Cp(T[i], Xa[i], Xc[i])
    V = f_Vol(M[i],rm)

    tabla.append([
        c,          # Corriente
        T[i],       # Temperatura
        Xa[i],      # Fracción másica agua
        Xc[i],      # Fracción másica crudo
        rm,         # Densidad mezcla
        Cp,         # Capacidad calorifica
        V           # Flujo volumétrico
    ])

df = pd.DataFrame(
    tabla,
    columns=[
        "Corriente",
        "T [°C]",
        "Xa [-]",
        "Xc [-]",
        "ρ mezcla [kg/m³]",
        "Cp [kJ/ton·K]",
        "V [KBPD]"
    ]
)

In [41]:
df["M [ton/d]"] = M
print(df.columns.tolist())

['Corriente', 'T [°C]', 'Xa [-]', 'Xc [-]', 'ρ mezcla [kg/m³]', 'Cp [kJ/ton·K]', 'V [KBPD]', 'M [ton/d]']


In [42]:
df = df[[
    "Corriente",
    "V [KBPD]",
    "M [ton/d]",
    "Xa [-]",
    "Xc [-]",
    "T [°C]",
    "ρ mezcla [kg/m³]",
    "Cp [kJ/ton·K]"
]]

print(df)

    Corriente  V [KBPD]  M [ton/d]  Xa [-]  Xc [-]  T [°C]  ρ mezcla [kg/m³]  \
0           1    117.40   18092.70  0.9423  0.0577   62.78            975.39   
1           2    127.56   19572.70  0.9030  0.0970   62.74            971.12   
2           3    128.80   19572.70  0.9030  0.0970   78.89            961.79   
3           4     21.04    3019.56  0.3717  0.6283   76.67            908.36   
4           5     21.04    3019.56  0.3717  0.6283   76.67            908.36   
5           6     21.17    3019.56  0.3717  0.6283   86.11            902.61   
6           7      7.40    1018.10  0.0055  0.9945   82.22            870.77   
7           8      7.32    1018.10  0.0055  0.9945   65.52            880.72   
8           9      7.32    1018.10  0.0055  0.9945   65.52            880.72   
9          10      7.32    1018.10  0.0055  0.9945   65.52            880.72   
10         11    107.58   16553.14  0.9999  0.0001   76.67            973.86   
11         12    107.58   16553.14  0.99

In [43]:
df.to_excel("Tabla_resultados.xlsx", index=False)